# SigLIP 2 Vision-Language Pipeline — DIMER multi-capability tutorial

**Profile:** `MULTI-CAPABILITY`  
**Notebook specification:** DIMER Notebook Specification v1.0

This notebook demonstrates the public `siglip2_pipeline` API with the pinned
`google/siglip2-base-patch16-224` model for five supported operations:
zero-shot classification, image embeddings, text embeddings, image-text
similarity, and text-to-image retrieval.

SigLIP 2 maps images and text into a shared representation space. Classification
here compares an image with prompt-wrapped candidate labels and returns one
independent sigmoid score per label. Embeddings are L2-normalized
representations; similarity and retrieval use cosine similarity through the dot
product of normalized vectors.

**No gradient training, fine-tuning, in-context conditioning, or fitted
preprocessing state occurs in this workflow.** The upstream model supplies the
pretrained weights and processor. This repository adds immutable checkpoint
pinning and integrity verification, safe local loading, a stable public inference
API, output contracts, and machine-readable provenance.

By the end of the notebook you will be able to:

- install the frozen reference environment and identify the effective runtime;
- acquire and verify the exact pinned SigLIP 2 checkpoint;
- run all five public inference capabilities through the repository API;
- interpret SigLIP sigmoid and cosine-similarity scores without treating them as calibrated probabilities;
- run a small labelled sanity evaluation and compare it with a trivial baseline;
- optionally upload a new image in Colab or provide a local image path in Jupyter;
- export classification, retrieval, similarity, embedding, metric, and provenance outputs.

**This notebook does not demonstrate:** object detection, semantic segmentation,
OCR, caption generation, fine-tuning, calibrated probabilities, a universal
classification threshold, or production HTTP/DIMER worker serving.

References: [repository README](../README.md) · [model card](../MODEL_CARD.md) ·
[upstream model](https://huggingface.co/google/siglip2-base-patch16-224) ·
[SigLIP 2 paper](https://arxiv.org/abs/2502.14786)


## Prerequisites

- **Runtime:** Python 3.12 is the supported reference runtime. CPU is supported
  for the default path; CUDA is used automatically when available.
- **Model access:** the verified weight file is 1,500,800,904 bytes. If no
  verified offline copy is present, the first run needs network access to
  Hugging Face at the pinned revision.
- **Data:** the default path generates deterministic local synthetic images, so
  no private user data is required.
- **BYOD:** optional and disabled by default. In Colab, an upload dialog appears
  only when `ENABLE_BYOD=True`. Uploaded files remain in the notebook runtime;
  this pipeline rejects HTTP(S) image URLs and does not itself send image
  contents to an external inference service.
- **Privacy:** do not upload confidential, restricted, or sensitive images to a
  hosted notebook environment unless you are authorized to place them there.

The pipeline converts images to RGB and uses the pinned processor's 224×224
image contract. Model-bound text is lowercased before tokenization and uses a
maximum text length of 64 tokens. Candidate labels and retrieval queries must be
non-empty strings; retrieval `top_k` must be at least 1. No repository-level
input file-size ceiling is imposed, so users remain responsible for choosing
images appropriate to available memory.

The tutorial does not claim measured runtime or memory performance for any
particular Colab hardware.


## 1. Bootstrap the repository and frozen environment

This stage clones the canonical repository only when the notebook is not already
running from a checkout. It installs the fully pinned reference dependency graph
first, then installs the local package with dependency resolution disabled.

A successful cell ends by printing the repository root. Package installation
occurs before importing PyTorch or Transformers, so no package-replacement
restart boundary is crossed in the primary path.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/siglip2-vision-language-pipeline.git"
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = Path("/content/siglip2-vision-language-pipeline")
    if not ROOT.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.lock.txt"],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--no-build-isolation",
        "-e",
        ".",
    ],
    check=True,
)
print(f"repository root: {ROOT.resolve()}")


## 2. Verify runtime identity

The notebook reports the principal runtime and model-framework versions before
model execution. Device selection is explicit: CUDA when available, otherwise
CPU. The model is loaded with the framework's default inference dtype; this
tutorial does not request mixed precision, quantization, FlashAttention,
compilation, or custom deterministic kernels.

No random split, sampling, ensemble, stochastic decoding, or random model
initialization is used, so the tutorial does not require an RNG seed for its
reported metrics. Small floating-point differences may still occur across
hardware, accelerator kernels, or library builds; reproducibility here means the
same pinned inputs/configuration and workflow, not bitwise-identical outputs
across every device.


In [ ]:
import platform
from importlib.metadata import version

import numpy as np
import torch
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", version("transformers"))
print("Hugging Face Hub:", version("huggingface-hub"))
print("Device:", DEVICE)
print("Quantization: none")
print("Compilation: none")


## 3. Resolve and verify the pinned model

`load_pipeline()` is the repository's public construction path. It resolves the
single supported upstream checkpoint at an immutable revision, verifies the
expected `model.safetensors` byte size and SHA-256, rejects unsafe weight
formats, and loads only from the verified local snapshot with remote code
disabled.

The output below is the effective model identity used by this run. An integrity
or network failure raises an exception rather than silently substituting another
checkpoint.


In [ ]:
from siglip2_pipeline import (
    MODEL_ID,
    MODEL_REVISION,
    build_provenance,
    load_pipeline,
    write_provenance,
)

pipe = load_pipeline(device=DEVICE)
run_provenance = build_provenance(pipeline=pipe)

print("Model ID:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Checkpoint source:", run_provenance["model"].get("checkpoint_source"))
print("Manifest verified:", run_provenance["model"].get("manifest_verified"))
print("Weight SHA-256:", run_provenance["model"]["weight_sha256"])
print("Weight bytes:", run_provenance["model"]["weight_size_bytes"])
print("Loaded device:", run_provenance["inference"].get("device"))


## 4. Generate the deterministic default sample

The repository's sample generator creates three 32×32 RGB PPM images:
`red_square.ppm`, `green_circle.ppm`, and `blue_triangle.ppm`. These are
**synthetic tutorial/smoke assets**, not benchmark data and not evidence of
production accuracy. Their purpose is to make the inference contracts and
output plumbing falsifiable without requiring private data.

The model processor converts the source images to the pinned 224×224 input
representation internally. The source files remain unchanged.


In [ ]:
import hashlib

SAMPLE_DIR = ROOT / "outputs" / "sample-data"
subprocess.run(
    [
        sys.executable,
        "examples/sample-data/generate_samples.py",
        "--output-dir",
        str(SAMPLE_DIR),
    ],
    check=True,
)

images = [
    SAMPLE_DIR / "red_square.ppm",
    SAMPLE_DIR / "green_circle.ppm",
    SAMPLE_DIR / "blue_triangle.ppm",
]
expected_labels = ["red square", "green circle", "blue triangle"]
candidate_labels = [*expected_labels, "abstract geometric shape"]

for path in images:
    with Image.open(path) as image:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        print(path.name, "mode=", image.mode, "size=", image.size, "sha256=", digest)


## 5. Zero-shot classification and sanity evaluation

`zero_shot_classify(image, labels)` wraps each candidate label with the
repository's prompt template, runs the real SigLIP 2 inference path, applies an
independent sigmoid to each image-text logit, and returns scores sorted
descending. Candidate-label wording and the prompt template are part of the
model input, so changing them can change rankings.

**Score semantics:** these sigmoid scores are not calibrated probabilities and
do not need to sum to 1. The default tutorial decision rule is `argmax` over the
candidate-label scores. A deployment-specific threshold or abstention rule, if
needed, is owned by the downstream application and must be calibrated on
representative labelled data.

There is no training/support split in this zero-shot workflow, so training-class
coverage validation is not applicable: the supplied candidate labels define the
classes evaluated at inference time.

Because the synthetic image names define tutorial ground truth, this section
reports **top-1 accuracy**: the fraction of the three images whose highest-scored
candidate label matches the intended label. It is a single tiny deterministic
sample with no dispersion estimate, not a benchmark or estimate of
generalization. The trivial baseline predicts one fixed class for all three
balanced examples, giving accuracy 1/3.


In [ ]:
from dataclasses import asdict

classification_rows = []
correct = 0

for image_path, expected in zip(images, expected_labels, strict=True):
    scores = pipe.zero_shot_classify(image_path, candidate_labels)
    predicted = scores[0].label
    correct += int(predicted == expected)
    classification_rows.append(
        {
            "image": image_path.name,
            "expected_label": expected,
            "predicted_label": predicted,
            "scores": [asdict(item) for item in scores],
        }
    )
    print(image_path.name, "expected=", expected, "predicted=", predicted)

top1_sanity_accuracy = correct / len(images)
majority_baseline_accuracy = 1 / len(expected_labels)

print(f"Tutorial top-1 sanity accuracy: {top1_sanity_accuracy:.3f}")
print(f"Fixed-class baseline accuracy: {majority_baseline_accuracy:.3f}")


## 6. Image and text embeddings

`embed_image()` and `embed_text()` return L2-normalized vectors from the shared
SigLIP representation space. Image embeddings are one vector per input image;
text embeddings are one vector per supplied text string.

Embeddings are **representations, not predictions**. They have no intrinsic
accuracy metric. Their usefulness must be evaluated on a downstream labelled
task such as retrieval relevance, classification, clustering quality, or another
domain-specific criterion. Missing-data semantics are not applicable to these
RGB image/text inputs; the encoder sees exactly the decoded image and supplied
text after repository preprocessing.


In [ ]:
image_embeddings = pipe.embed_image(images)
text_embeddings = pipe.embed_text(expected_labels)

print("Image embedding matrix:", image_embeddings.shape)
print("Text embedding matrix:", text_embeddings.shape)
print(
    "Image L2 norms:",
    np.round(np.linalg.norm(image_embeddings, axis=1), 6).tolist(),
)
print(
    "Text L2 norms:",
    np.round(np.linalg.norm(text_embeddings, axis=1), 6).tolist(),
)


## 7. Image-text similarity

`similarity(images, texts)` returns a matrix with one row per image and one
column per text. Values are cosine similarities because both embedding sets are
L2-normalized. Similarity values are ranking/association scores, not calibrated
probabilities.

The synthetic diagonal is easy to inspect, but any apparent alignment here is
only tutorial evidence.


In [ ]:
similarity = pipe.similarity(images, expected_labels)

print("Rows:", [path.name for path in images])
print("Columns:", expected_labels)
print(np.array2string(similarity, precision=4))


## 8. Text-to-image retrieval

`retrieve(query, images, top_k)` embeds the query and candidate images, ranks
cosine similarity descending, and returns source indexes plus scores. Query
wording is part of the inference configuration, so retrieval rankings are
prompt-dependent. The pipeline does not ship a universal relevance threshold.

For this labelled synthetic sample, **recall@1** is the fraction of queries whose
intended image is ranked first. Each query has exactly one intended image. As
with classification accuracy above, the result is a single deterministic
tutorial sanity metric with no dispersion estimate, not benchmark evidence.


In [ ]:
retrieval_rows = []
retrieval_hits_at_1 = 0

for query, expected_image in zip(expected_labels, images, strict=True):
    hits = pipe.retrieve(query, images, top_k=len(images))
    top_image = images[hits[0].index]
    retrieval_hits_at_1 += int(top_image.name == expected_image.name)
    retrieval_rows.append(
        {
            "query": query,
            "expected_image": expected_image.name,
            "hits": [
                {
                    "rank": rank,
                    "index": hit.index,
                    "filename": images[hit.index].name,
                    "score": hit.score,
                }
                for rank, hit in enumerate(hits, start=1)
            ],
        }
    )
    print(query, "->", top_image.name)

retrieval_recall_at_1 = retrieval_hits_at_1 / len(expected_labels)
print(f"Tutorial retrieval recall@1: {retrieval_recall_at_1:.3f}")


## 9. Optional BYOD / new user image

The default notebook path does **not** open an upload dialog. Set
`ENABLE_BYOD=True` to exercise a real new-image path.

Expected input: one local image file that Pillow can decode. In Colab, choose the
file from the upload dialog. In another Jupyter environment, set `BYOD_PATH` to
an existing local file. The notebook validates that the file exists and can be
decoded before model execution, reports its original mode/dimensions, then uses
the repository's production-facing preprocessing/inference path.

Candidate labels below are part of the inference configuration; edit them for
your domain. Keep prompts concise because model-bound text uses the 64-token
processor contract. The repository lowercases text sent to the model while
preserving caller-facing labels.

This path performs classification and image embedding on genuinely new user
input. It does not send image contents to a hosted inference API; a hosted
Colab runtime itself remains an external computing environment.


In [ ]:
ENABLE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ""  # @param {type:"string"}
BYOD_LABELS = ["flooded street", "normal road", "fallen electrical pole"]

byod_image = None
if ENABLE_BYOD:
    selected_path = None

    try:
        from google.colab import files as colab_files  # type: ignore

        uploaded = colab_files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one image file for this tutorial path.")
        uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
        upload_dir = ROOT / "outputs" / "byod"
        upload_dir.mkdir(parents=True, exist_ok=True)
        selected_path = upload_dir / Path(uploaded_name).name
        selected_path.write_bytes(uploaded_bytes)
    except ModuleNotFoundError:
        if not BYOD_PATH:
            raise RuntimeError(
                "Outside Colab, set BYOD_PATH to one existing local image before "
                "enabling the BYOD path."
            ) from None
        selected_path = Path(BYOD_PATH).expanduser()

    if not selected_path.is_file():
        raise FileNotFoundError(f"BYOD image does not exist: {selected_path}")

    try:
        with Image.open(selected_path) as image:
            image.verify()
        with Image.open(selected_path) as image:
            print("BYOD image:", selected_path.name, "mode=", image.mode, "size=", image.size)
    except Exception as exc:
        raise ValueError(f"BYOD file is not a decodable image: {selected_path}") from exc

    if not BYOD_LABELS or any(not label.strip() for label in BYOD_LABELS):
        raise ValueError("BYOD_LABELS must contain non-empty candidate labels.")

    byod_image = selected_path
    byod_scores = pipe.zero_shot_classify(byod_image, BYOD_LABELS)
    byod_embedding = pipe.embed_image([byod_image])

    print("BYOD top label:", byod_scores[0].label)
    print("BYOD embedding shape:", byod_embedding.shape)
else:
    print("BYOD disabled; default sample path remains non-interactive.")


## 10. Default new-data inference

A release tutorial should demonstrate inference on data that is distinct from
the small evaluation set. To keep the default path non-interactive, this cell
creates a deterministic `yellow_square.ppm` that was not part of the preceding
three-image evaluation set, validates it, and classifies it through the same
public API.

This is still synthetic demonstration evidence; it proves the new-input code
path, not deployment accuracy.


In [ ]:
NEW_DATA_DIR = ROOT / "outputs" / "new-data"
NEW_DATA_DIR.mkdir(parents=True, exist_ok=True)
new_image_path = NEW_DATA_DIR / "yellow_square.ppm"

new_image = Image.new("RGB", (32, 32), "white")
pixels = new_image.load()
for y in range(8, 24):
    for x in range(8, 24):
        pixels[x, y] = (255, 255, 0)
new_image.save(new_image_path)

with Image.open(new_image_path) as image:
    print("New image:", new_image_path.name, "mode=", image.mode, "size=", image.size)

new_labels = ["yellow square", "blue circle", "red triangle", "abstract geometric shape"]
new_data_scores = pipe.zero_shot_classify(new_image_path, new_labels)
print("New-data top label:", new_data_scores[0].label)


## 11. Export machine-readable outputs and provenance

This stage writes stable files for every demonstrated capability. Embedding
archives include identifiers beside vectors so downstream consumers can map each
row back to its source image or text. Provenance records the exact model,
immutable revision, verified checkpoint identity, runtime packages, device, and
inference semantics.

No model artifact is produced because this workflow performs pretrained
inference only; artifact-export and fresh-reload requirements therefore do not
apply.


In [ ]:
import csv
import json

OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)

(OUTPUT / "classification.json").write_text(
    json.dumps(classification_rows, indent=2) + "\n",
    encoding="utf-8",
)
(OUTPUT / "retrieval.json").write_text(
    json.dumps(retrieval_rows, indent=2) + "\n",
    encoding="utf-8",
)
(OUTPUT / "new_data_classification.json").write_text(
    json.dumps(
        {
            "image": new_image_path.name,
            "scores": [asdict(item) for item in new_data_scores],
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

with (OUTPUT / "similarity.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["image", *expected_labels])
    for image_path, row in zip(images, similarity, strict=True):
        writer.writerow([image_path.name, *[float(value) for value in row]])

np.savez_compressed(
    OUTPUT / "image_embeddings.npz",
    image_ids=np.asarray([path.name for path in images]),
    vectors=image_embeddings,
)
np.savez_compressed(
    OUTPUT / "text_embeddings.npz",
    text_ids=np.asarray(expected_labels),
    vectors=text_embeddings,
)

sample_sha256 = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in images
}

metrics = {
    "evidence_type": "synthetic_tutorial_sanity_only",
    "sample_sha256": sample_sha256,
    "classification": {
        "metric": "top1_accuracy",
        "estimation": "single deterministic three-example tutorial sample",
        "value": top1_sanity_accuracy,
        "fixed_class_baseline": majority_baseline_accuracy,
    },
    "retrieval": {
        "metric": "recall_at_1",
        "estimation": "single deterministic three-query tutorial sample",
        "value": retrieval_recall_at_1,
    },
}
(OUTPUT / "metrics.json").write_text(
    json.dumps(metrics, indent=2) + "\n",
    encoding="utf-8",
)

if ENABLE_BYOD and byod_image is not None:
    (OUTPUT / "byod_classification.json").write_text(
        json.dumps(
            {
                "image": byod_image.name,
                "scores": [asdict(item) for item in byod_scores],
            },
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )
    np.savez_compressed(
        OUTPUT / "byod_image_embedding.npz",
        image_ids=np.asarray([byod_image.name]),
        vectors=byod_embedding,
    )

write_provenance(OUTPUT / "provenance.json", pipeline=pipe)

expected_outputs = [
    "classification.json",
    "image_embeddings.npz",
    "metrics.json",
    "new_data_classification.json",
    "provenance.json",
    "retrieval.json",
    "similarity.csv",
    "text_embeddings.npz",
]
missing_outputs = [name for name in expected_outputs if not (OUTPUT / name).is_file()]
if missing_outputs:
    raise RuntimeError(f"Missing expected tutorial outputs: {missing_outputs}")

print("Wrote:")
for name in expected_outputs:
    print(" -", name)


## Troubleshooting

- **Model download or integrity failure:** confirm network access to Hugging Face
  or provide the exact verified offline snapshot. Do not bypass SHA-256/size
  verification and do not substitute an unpinned checkpoint.
- **Out-of-memory error:** restart the runtime, use CPU if GPU memory is
  insufficient, and reduce the number of images processed per call. The
  tutorial makes no fixed memory-usage claim.
- **CUDA unavailable:** this is not fatal; the default code falls back to CPU
  and prints the selected device.
- **BYOD path error:** in Colab, enable BYOD and upload exactly one image. Outside
  Colab, set `BYOD_PATH` to an existing local image. Decode failures identify the
  rejected file before inference.
- **Unexpected scores:** verify candidate-label and query wording first. SigLIP
  scores are prompt-dependent and uncalibrated; a successful execution is not
  evidence that the prompts are appropriate for your domain.


## Interpretation, limits, and next steps

A successful top-to-bottom run demonstrates that this repository can install its
frozen reference environment, resolve and integrity-check the one pinned SigLIP
2 checkpoint, exercise all five public inference operations, produce
machine-readable outputs with identifiers, and record run provenance.

It **does not** establish production fitness, domain accuracy, fairness,
robustness, calibration, operational latency, or a universal classification or
retrieval threshold. The displayed accuracy and recall@1 values come from three
synthetic geometric images and are only sanity evidence. Sigmoid scores and
cosine similarities remain uncalibrated ranking/association scores.

For a real deployment, evaluate representative labelled images from the target
domain, define the actual candidate-label/prompt policy, measure relevant
failure modes and subgroup behavior, calibrate any thresholds or abstention
rules, and benchmark runtime on the intended serving hardware.

Useful next experiments are to enable BYOD with a non-sensitive image, compare
prompt variants, evaluate retrieval on a labelled real-image collection, and
measure whether downstream task performance from exported embeddings is adequate
for the intended application.
